In [39]:
import os
import certifi
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [40]:
from langchain.agents import create_react_agent, AgentExecutor


In [41]:
#========================================
# LOAD ENVIRONMENT VARIABLES
#========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [42]:
search_tool = TavilySearchResults(max_results=3)

In [43]:
@tool
def get_weather_data(city:str) -> str:
    """
    Get the current weather data for a given city
    """

    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current"  not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"Temperature: {data['current']['temperature']}°C, \n"
        f"Weather: {data['current']['weather_descriptions'][0]} \n"
        f"Humidity: {data['current']['humidity']}%, \n"
    )

In [59]:
result=search_tool.invoke("Give me the latest news on AI")
result
print(get_weather_data.invoke({"city": "Chennai"}))

Could not fetch weather data for Chennai


In [45]:
#-------------------------------
# LLM
#-------------------------------

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [46]:
response = llm.invoke("What is the year is it")
response

AIMessage(content='It is currently 2022.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 13, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-264e8b5a-01db-4e82-a6ce-03ebcd5cb0d6-0')

In [47]:
#PROMPT
#=========================================

prompt = hub.pull("hwchase17/react")

c:\Users\Dell\.conda\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [48]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [49]:
#=========================================
# Tools

#==============================

tools = [search_tool, get_weather_data]


In [50]:
#=========================================
# CREATE AGENT
#=============================================

agent = create_react_agent(
    llm=llm,
    prompt=prompt,
    tools=tools
)

In [53]:
#=========================================
#Executor
#=========================================

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [56]:
#=========================================
#RUN
#=========================================

response = agent_executor.invoke({
    "input":(
        "Find the capital of India"
        " and then find its current weather"
    )
})



> Entering new AgentExecutor chain...
I should first search for the capital of India and then use the weather data tool to find its current weather.
Action: tavily_search_results_json
Action Input: "capital of India"[{'url': 'https://www.usatoday.com/story/news/world/2023/05/24/what-is-the-capital-of-india/70195720007', 'content': "## What is the capital of India?\n\nThe capital of India is New Delhi, located in the north-central part of the country to the west of the Yamuna River.\n\nCalcutta (now Kolkata, the capital of West Bengal) was the country’s capital until 1911 when King George V declared Delhi the new capital and construction of New Delhi began.\n\nWhen the national government achieved independence in 1947, New Delhi became the capital.\n\nMumbai, the state capital of Maharashtra, is often considered the financial capital of India because of its role in the national and international economy.\n\n## How many states are in India?\n\nIndia is home to 28 states, each with its 

In [57]:
print(response["output"])

The capital of India is New Delhi, but the current weather data for New Delhi is not available at the moment.
